# Practice run of analysing/testing different models on the UNSW_NB15 dataset, before trying Deep Learning.

Prior research suggests this is a largely non-linear, less separable dataset so deep learning may be necessary, but I will try simpler, more interpretable models first for the sake of completeness, and to gain Variable Importances

Let's load our packages and data

In [ ]:
# Check and Install Python 3.9
!apt-get update
!apt-get install -y python3.9

# Install CUDA and NVIDIA Drivers (if not already done)
!wget https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2004/x86_64/cuda-ubuntu2004.pin
!mv cuda-ubuntu2004.pin /etc/apt/preferences.d/cuda-repository-pin-600
!wget https://developer.download.nvidia.com/compute/cuda/12.3.0/local_installers/cuda-repo-ubuntu2004-12-3-local_12.3.0-535.86.05-1_amd64.deb
!dpkg -i cuda-repo-ubuntu2004-12-3-local_12.3.0-535.86.05-1_amd64.deb
!apt-key add /var/cuda-repo-ubuntu2004-12-3-local/7fa2af80.pub
!apt-get update
!apt-get -y install cuda-toolkit-12-3

# Install NVIDIA Drivers (if not already done)
!apt-get install -y nvidia-driver-535

# Install cuDNN (if not already done)
!wget https://developer.download.nvidia.com/compute/redist/cudnn/v8.9.3/cudnn-linux-x86_64-v8.9.3.28_cuda12-archive.tar.xz # Update with actual cuDNN version
!tar -xvf cudnn-linux-x86_64-v8.9.3.28_cuda12-archive.tar.xz
!cp -a cudnn-linux-x86_64-v8.9.3.28_cuda12-archive/include/. /usr/local/cuda/include/
!cp -a cudnn-linux-x86_64-v8.9.3.28_cuda12-archive/lib64/. /usr/local/cuda/lib64/

# Install RAPIDS
!pip install --index-url https://pypi.nvidia.com --no-cache-dir \
    rapidsai==23.02 \
    cudf==23.02 \
    cuml==23.02 \
    cugraph==23.02 \
    dask-cudf==23.02 \
    rmm==23.02 -f
!pip install --no-deps scikit-learn==1.2.2 pandas==2.0.3 numpy==1.23.5 matplotlib==3.7.1 seaborn==0.12.2 --force-reinstall

import os
os.environ['NUMBAPRO_NVVM'] = '/usr/local/cuda/nvvm/lib64/libnvvm.so'
os.environ['NUMBAPRO_LIBDEVICE'] = '/usr/local/cuda/nvvm/libdevice/'
os.environ['LD_LIBRARY_PATH'] = os.environ.get('LD_LIBRARY_PATH', '') + ':/usr/local/cuda/lib64'

import sys
sys.path.append('/usr/local/lib/python3.10/dist-packages/')

print("Installation and setup complete.")

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,071 kB]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Ign:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy Release [5,713 B]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy Release.gpg [793 B]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2

Installation and setup complete.


In [ ]:
#import packages:

import os
os.environ['NUMBAPRO_NVVM'] = '/usr/local/cuda/nvvm/lib64/libnvvm.so'
os.environ['NUMBAPRO_LIBDEVICE'] = '/usr/local/cuda/nvvm/libdevice/'

from google.colab import drive

try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

if IN_COLAB:
  # Check if drive is mounted by looking for the mount point in the file system.
  # This is a more robust approach than relying on potentially internal variables.
  import os
  if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
  else:
    print("Google Drive is already mounted.")
else:
  print("Not running in Google Colab. Drive mounting skipped.")

from IPython import get_ipython
from IPython.display import display
import cudf
import cuml
from cuml.preprocessing import StandardScaler
from cuml.model_selection import StratifiedKFold, GridSearchCV
from cuml.linear_model import LogisticRegression
import pandas as pd
import sklearn as sk
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from tqdm import tqdm


print("New run: Packages loaded")

Google Drive is already mounted.


ModuleNotFoundError: No module named 'cuml'

In [ ]:
#if using colabs - will need to first mount your drive

#change these for different users
test_set_filepath = '/content/drive/MyDrive/Colab_Notebooks/Data/UNSW_NB15_testing-set.parquet'
training_set_filepath = '/content/drive/MyDrive/Colab_Notebooks/Data/UNSW_NB15_training-set.parquet'

# Import the two CSV files
test_set = pd.read_parquet(test_set_filepath)
train_set = pd.read_parquet(training_set_filepath)

print("Data loaded")


The next cell does some basic analysis, and one hot encodes some of the features:

In [ ]:
"""
#print number of records in our data
print(f"Number of records in training set: {len(train_set)}")
print(f"Number of records in test set: {len(test_set)}")

#lets see which ones are categorical etc
print(f'''
The columns and datatypes are:
{train_set.dtypes}
''')

print("Categorical Columns are :", categorical_cols)

#print out number of classifications
print(f"Number of categories in 'label' category: {len(train_set['label'].unique())}")

#print out labels
print(f"Labels: {train_set['label'].unique()}")

#print out how many unique values we have for each categorical variable - if we have too many we may need an embeddings layer
for col in categorical_cols:
    print(f"Number of categories in '{col}' category: {len(train_set[col].unique())}")

"""



In [ ]:

def preprocess_data(data_set):
  """
  Function to preprocess data. One hot encodes the top 6 most common values for 'proto'. NOTE - to self, might need to do this grouping for other categoricals?
  And turns boolean columns into 1s and 0s.

  Args:
  data_set (dataframe) : test or train set to be processed

  Retuns:
  data_set (dataframe) : processed data set

  """

  #we aer going to set this as a binary classification task for now. This is because Logistic Regression struggles with multiclass, and these interpretable models are only EDA really
  if 'attack_cat' in data_set.columns.tolist():
    data_set = data_set.drop('attack_cat', axis = 1)

  #there seems to be over 100 possible values of proto - lets see how common they all are
  if 'proto' in data_set.columns.tolist():
    category_percentages = data_set['proto'].value_counts(normalize=True) * 100

    #define a dict of the categories and their percentages of occurence. what we want to do here is group any that occur less than 0.5% of the time, into an 'other' category
    category_percentages_dict = category_percentages.to_dict()
    #we can then print this to view the distributions ^

    # After looking at the distributions of the possible values for Proto, only the top 6 occur more than 0.5% of the time - hence all others are very rare
    # So we get the top 6 most common values. We have to hardcode in this value of top 6
    top_6_categories = category_percentages.head(6).index.tolist()

    #we now have a list of values that we want to one hot encode. we want to simply group the others into an 'other column'
    data_set['proto_grouped'] = data_set['proto'].applymap(lambda x: x if x in top_6_categories else 'other')

    #now we one hot encode this column
    data_set = cudf.get_dummies(data_set, columns=['proto_grouped'], prefix='proto_grouped') # Use cudf.get_dummies

    #drop the original columns if still present
    data_set = data_set.drop('proto', axis=1)

  if 'proto_grouped' in data_set.columns.tolist():
      data_set = data_set.drop(['proto_grouped'], axis=1)

  # List only the categorical columns (object types)
  categorical_cols = data_set.select_dtypes(include=['category']).columns.tolist()

  #one hot encode all remaining categorical columns
  data_set = cudf.get_dummies(data_set, columns=categorical_cols, prefix_sep='_')  # Use cudf.get_dummies

  #encode all binary data as 1s and 0s
  binary_cols = data_set.select_dtypes(include=['bool']).columns

  #convert to int
  data_set[binary_cols] = data_set[binary_cols].astype(int)

  print(f"Data set preprocessed, columns = {data_set.columns.tolist()}")

  return data_set

train_set = preprocess_data(train_set)




NOTE TO SELF -
1. THIS IS FOR BINARY CLASSIFICATION, WE WANT MULTICLASS EVENTUALLY, BUT FOR NOW WE WILL JUST DO BN


Based on the high number of columns in the Proto column, we may want to consider an Embeddings layer with the Deep Learning that we plan to undertake later. However since DT/RF perform somewhat poorly on sparse vector datasets (like one hot encoded ones) we will group all the extremely rare categories into an 'other'.


In [ ]:
def run_models(model_type, test_set = test_set, train_set = train_set):
  """
  Runs LR, DT or RF model on dataframe
  """

  #drop label and define list of out targets
  X_train = train_set.drop('label', axis=1)
  y_train = train_set['label']

  #do the same for test set
  X_test = test_set.drop('label', axis=1)
  y_test = test_set['label']

  # List only the categorical columns (object types)
  categorical_cols = train_set.select_dtypes(include=['category']).columns.tolist()


#=============================== LR ========================================#

  if model_type.upper() == 'LR':

    # Define the pipeline
    pipeline = Pipeline([
          ('scaler', StandardScaler()),
          ('classifier', LogisticRegression(max_iter=1000))
      ])

    # Define the hyperparameter grid
    param_grid = {
        'classifier__C': [0.01, 0.1, 1, 10, 100],
        'classifier__penalty': ['l1', 'l2'],
        'classifier__solver': ['liblinear']  # Suitable for smaller datasets
    }

    # Set up cross-validation strategies
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

    print("KFolds defined, beginning nested cross-validation.")

    # Outer loop (on training data)
    outer_scores = []

    #outer loop
    for train_index, val_index in tqdm(outer_cv.split(X_train, y_train)):
        X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
        y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

        # Set up GridSearchCV with the pipeline (inner loop)
        grid_search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=inner_cv,
            scoring='roc_auc',  # Use appropriate scoring for your task
        )

        # Fit GridSearchCV on the training fold
        grid_search.fit(X_train_fold, y_train_fold)

        # Evaluate on the validation fold
        score = grid_search.score(X_val_fold, y_val_fold)
        outer_scores.append(score)

    # Print the average outer score (ROC AUC) on training data
    print(f"Average Validation ROC AUC: {np.mean(outer_scores)}")

    # Evaluate the best model on the held-out test set
    best_model = grid_search.best_estimator_
    test_score = best_model.score(X_test, y_test)
    print(f"Test ROC AUC: {test_score}")

#=============================== DT ========================================#

  if model_type.upper() == 'DT':

    param_grid_dt = {
    'max_depth': [3, 5, 10],             # List of max depth values
    'min_samples_split': [2, 10, 20],    # List of minimum samples to split
    'min_samples_leaf': [1, 5, 10]       # List of minimum samples per leaf
    }

    pass
#=============================== RF ========================================#

  if model_type.upper() == 'RF':

    param_grid_rf = {
    'n_estimators': [50, 100, 200],      # List of number of trees to use
    'max_depth': [5, 10, 20],            # List of maximum depths for trees
    'min_samples_split': [2, 10, 20]     # List of minimum samples for splitting a node
    }

    pass

run_models('LR')

KFolds defined, beginning nested cross-validation.


0it [00:00, ?it/s]